# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [48]:
# This cell is for CODE (numbers, a query, a check).

import duckdb
from google.colab import userdata

MONTH = "2026-03"          # assigned mid-panel month
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_TABLE = f"{REL}/fact_content_daily_performance/month={MONTH}/*.parquet"

con = duckdb.connect()
token = userdata.get("HF_TOKEN")

con.sql(f"""
    CREATE OR REPLACE SECRET hf_token (
        TYPE huggingface,
        TOKEN '{token}'
    )
""")

api = HfApi(token=userdata.get("HF_TOKEN"))
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")

for f in files:
    if "client" in f.lower():
        print(f)

secret_check = con.sql("SELECT * FROM duckdb_secrets()").df()
print(f"Working month: {MONTH}")

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


dim_clients.parquet
Working month: 2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [44]:
# This cell is for CODE (numbers, a query, a check).

q_grain = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_triples
FROM read_parquet('{FACT_TABLE}')
"""
con.sql(q_grain).show()

q_span = f"""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT client_hash_id) AS n_clients,
    COUNT(DISTINCT content_hash_id) AS n_content_items,
    MIN(report_date) AS earliest_date,
    MAX(report_date) AS latest_date,
    COUNT(DISTINCT report_date) AS n_distinct_dates
FROM read_parquet('{FACT_TABLE}')
"""
con.sql(q_span).show()

q_avail = f"""
SELECT
    COUNT(*) AS total_rows,
    SUM((gsc_data_available IS TRUE)::INT)               AS gsc_available_rows,
    SUM((ga4_data_available IS TRUE)::INT)                AS ga4_available_rows,
    SUM((gsc_data_available IS TRUE AND ga4_data_available IS TRUE)::INT) AS both_available_rows
FROM read_parquet('{FACT_TABLE}')
"""
con.sql(q_avail).show()

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────────┐
│ total_rows │ distinct_triples │
│   int64    │      int64       │
├────────────┼──────────────────┤
│    9841378 │          9841378 │
└────────────┴──────────────────┘

┌─────────┬───────────┬─────────────────┬───────────────┬─────────────┬──────────────────┐
│ n_rows  │ n_clients │ n_content_items │ earliest_date │ latest_date │ n_distinct_dates │
│  int64  │   int64   │      int64      │     date      │    date     │      int64       │
├─────────┼───────────┼─────────────────┼───────────────┼─────────────┼──────────────────┤
│ 9841378 │        55 │          331437 │ 2026-03-01    │ 2026-03-31  │               31 │
└─────────┴───────────┴─────────────────┴───────────────┴─────────────┴──────────────────┘

┌────────────┬────────────────────┬────────────────────┬─────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │ both_available_rows │
│   int64    │       int128       │       int128       │       int128        │
├──────────

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [45]:
# This cell is for CODE (numbers, a query, a check).

q_features = f"""
WITH agg AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks)::DOUBLE / NULLIF(SUM(gsc_impressions), 0)             AS ctr,
        SUM(ga4_engaged_sessions)::DOUBLE / NULLIF(SUM(ga4_sessions), 0)      AS engagement_rate,
        SUM(ga4_total_engagement_sec)::DOUBLE / NULLIF(SUM(ga4_sessions), 0)  AS session_depth_sec,
        SUM(sessions_ai)::DOUBLE / NULLIF(
            SUM(sessions_organic + sessions_direct + sessions_referral
                + sessions_social + sessions_paid + sessions_ai), 0)         AS ai_traffic_pct,
        AVG((gsc_data_available IS TRUE)::INT)                                AS gsc_availability_rate,
        COUNT(*) AS n_days_reported
    FROM read_parquet('{FACT_TABLE}')
    GROUP BY client_hash_id, content_hash_id
)
SELECT * FROM agg
"""
features_df = con.sql(q_features).df()
print(features_df.shape)
features_df.describe()

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(331437, 8)


,ctr,engagement_rate,session_depth_sec,ai_traffic_pct,gsc_availability_rate,n_days_reported
count,176738.000000,90237.000000,90237.000000,82716.000000,331437.000000,331437.000000
mean,0.004594,0.025977,4.406660,0.010660,0.357092,29.693058
std,0.037760,0.110565,25.865938,0.075926,0.427416,4.738734
min,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,31.000000
50%,0.000000,0.000000,0.000000,0.000000,0.034483,31.000000
75%,0.002158,0.000000,1.000000,0.000000,0.903226,31.000000
max,1.000000,1.000000,1280.000000,1.000000,1.000000,31.000000


In [46]:
# This cell is for CODE (numbers, a query, a check).

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

feature_cols = ["ctr", "engagement_rate", "session_depth_sec", "ai_traffic_pct", "gsc_availability_rate"]

clean = features_df.dropna(subset=feature_cols).copy()
X = StandardScaler().fit_transform(clean[feature_cols])

# Honest baseline
km_honest = KMeans(n_clusters=5, random_state=42, n_init=10).fit(X)
score_honest = silhouette_score(X, km_honest.labels_)
print(f"Honest silhouette score: {score_honest:.4f}")

# Deliberate leak: feed the resulting cluster label back in as a "feature"
clean["leaked_cluster_id"] = km_honest.labels_
X_leaked = StandardScaler().fit_transform(clean[feature_cols + ["leaked_cluster_id"]])
km_leaked = KMeans(n_clusters=5, random_state=42, n_init=10).fit(X_leaked)
score_leaked = silhouette_score(X_leaked, km_leaked.labels_)
print(f"Leaked silhouette score:  {score_leaked:.4f}")

# Keep the honest number
print(f"\nDelta from leakage: {score_leaked - score_honest:+.4f}")
print("Dropping leaked_cluster_id — keeping the honest baseline as the real result.")
clean = clean.drop(columns=["leaked_cluster_id"])

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

Honest silhouette score: 0.6544
Leaked silhouette score:  0.7119

Delta from leakage: +0.0574
Dropping leaked_cluster_id — keeping the honest baseline as the real result.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [49]:
# This cell is for CODE (numbers, a query, a check).

print("Non-null counts per feature:")
print(features_df[feature_cols].count())

# Panel imbalance: does each client's GSC/GA4 history actually cover this month?
q_panel = f"""
SELECT
    MIN(gsc_data_start) AS earliest_gsc_start,
    MAX(gsc_data_start) AS latest_gsc_start,
    SUM((gsc_data_start > DATE '2026-03-01')::INT) AS clients_starting_after_march
FROM read_parquet('{REL}/dim_clients.parquet')
"""
con.sql(q_panel).show()

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Non-null counts per feature:
ctr                      176738
engagement_rate           90237
session_depth_sec         90237
ai_traffic_pct            82716
gsc_availability_rate    331437
dtype: int64
┌────────────────────┬──────────────────┬──────────────────────────────┐
│ earliest_gsc_start │ latest_gsc_start │ clients_starting_after_march │
│        date        │       date       │            int128            │
├────────────────────┼──────────────────┼──────────────────────────────┤
│ 2025-01-27         │ 2026-06-02       │                           15 │
└────────────────────┴──────────────────┴──────────────────────────────┘



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.